# Tabular + text (supplements) — no images, no CNN

Uses **`dataset/supplements/training_supp.csv`** (diagnosis text and confirm type) joined with **`training_input.csv`** on `isic_id` and **`training_gt.csv`** on `lesion_id` for the same multi-label targets as the image notebooks. This stays **separate** from EfficientNet and HiResCAM.

Good as a lightweight baseline or when you only have metadata + text.

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, hamming_loss
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

REPO_ROOT = Path("..").resolve()
SUPP_DIR = REPO_ROOT / "dataset" / "supplements"

SUP_PATH = SUPP_DIR / "training_supp.csv"
INP_PATH = SUPP_DIR / "training_input.csv"
GT_PATH = SUPP_DIR / "training_gt.csv"

LABEL_COLS = [
    "AKIEC", "BCC", "BEN_OTH", "BKL", "DF", "INF",
    "MAL_OTH", "MEL", "NV", "SCCKA", "VASC",
]

RANDOM_STATE = 42

In [6]:
sup = pd.read_csv(SUP_PATH)
inp = pd.read_csv(INP_PATH)
gt = pd.read_csv(GT_PATH)

df = sup.merge(inp, on="isic_id", how="inner").merge(gt, on="lesion_id", how="inner")
print("rows:", len(df), "columns:", len(df.columns))
df.head(2)

rows: 10480 columns: 31


,isic_id,diagnosis_full,diagnosis_confirm_type,invasion_thickness_interval,lesion_id,image_type,attribution,copyright_license,image_manipulation,age_approx,...,BCC,BEN_OTH,BKL,DF,INF,MAL_OTH,MEL,NV,SCCKA,VASC
0,ISIC_0051817,"Squamous cell carcinoma, Invasive",histopathology,NaN,IL_1612768,dermoscopic,MILK study team,CC-BY-NC,instrument only,70.0,...,0,0,0,0,0,0,0,0,1,0
1,ISIC_0073863,"Nevus, Reed",histopathology,NaN,IL_2547802,dermoscopic,MILK study team,CC-BY-NC,instrument only,5.0,...,0,0,0,0,0,0,0,1,0,0


In [7]:
text_col = "diagnosis_full"
cat_cols = [
    "diagnosis_confirm_type",
    "image_type",
    "sex",
    "site",
    "image_manipulation",
    "invasion_thickness_interval",
]
num_cols = [c for c in df.columns if c.startswith("MONET_")]
num_cols += [c for c in ("age_approx", "skin_tone_class") if c in df.columns]

for c in cat_cols + [text_col]:
    df[c] = df[c].fillna("").astype(str)
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

y = df[LABEL_COLS].values.astype(np.int32)
X = df[cat_cols + num_cols + [text_col]].copy()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE, shuffle=True
)
len(X_train), len(X_val)

(8908, 1572)

In [8]:
categorical_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False, max_categories=30),
        ),
    ]
)

numeric_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
)

text_pipe = Pipeline(
    steps=[
        ("tfidf", TfidfVectorizer(max_features=4000, ngram_range=(1, 2))),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("cat", categorical_pipe, cat_cols),
        ("num", numeric_pipe, num_cols),
        ("txt", text_pipe, text_col),
    ],
    remainder="drop",
)

clf = MultiOutputClassifier(
    LogisticRegression(max_iter=200, C=0.5, solver="lbfgs"),
    n_jobs=-1,
)

model = Pipeline([("prep", preprocess), ("clf", clf)])
model.fit(X_train, y_train)

proba = model.predict_proba(X_val)
# predict_proba returns list per label; stack to (n, n_labels)
p = np.vstack([proba[i][:, 1] for i in range(len(LABEL_COLS))]).T
pred = (p >= 0.5).astype(np.int32)

print("hamming loss:", hamming_loss(y_val, pred))
print("macro F1:", f1_score(y_val, pred, average="macro", zero_division=0))

hamming loss: 0.0008096229470275272
macro F1: 0.9420286332132829


In [ ]:
# Safer tabular split: group by lesion_id and keep a held-out test set
from sklearn.model_selection import GroupShuffleSplit

X_full = df[cat_cols + num_cols + [text_col]].copy()
y_full = df[LABEL_COLS].values.astype(np.int32)
groups = df["lesion_id"].astype(str).values

gss_outer = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_STATE)
train_val_idx, test_idx = next(gss_outer.split(X_full, y_full, groups=groups))

gss_inner = GroupShuffleSplit(n_splits=1, test_size=0.17647, random_state=RANDOM_STATE)
train_idx, val_idx = next(
    gss_inner.split(
        X_full.iloc[train_val_idx],
        y_full[train_val_idx],
        groups=groups[train_val_idx],
    )
)

X_train_g = X_full.iloc[train_val_idx].iloc[train_idx]
X_val_g = X_full.iloc[train_val_idx].iloc[val_idx]
X_test_g = X_full.iloc[test_idx]

y_train_g = y_full[train_val_idx][train_idx]
y_val_g = y_full[train_val_idx][val_idx]
y_test_g = y_full[test_idx]

print("Grouped split sizes:", len(X_train_g), len(X_val_g), len(X_test_g))
print(
    "Unique lesions:",
    X_train_g.index.size,
    X_val_g.index.size,
    X_test_g.index.size,
)

model_grouped = Pipeline([("prep", preprocess), ("clf", clf)])
model_grouped.fit(X_train_g, y_train_g)

val_proba = model_grouped.predict_proba(X_val_g)
val_p = np.vstack([val_proba[i][:, 1] for i in range(len(LABEL_COLS))]).T
val_pred = (val_p >= 0.5).astype(np.int32)

print("Grouped val hamming loss:", hamming_loss(y_val_g, val_pred))
print("Grouped val macro F1:", f1_score(y_val_g, val_pred, average="macro", zero_division=0))

test_proba = model_grouped.predict_proba(X_test_g)
test_p = np.vstack([test_proba[i][:, 1] for i in range(len(LABEL_COLS))]).T
test_pred = (test_p >= 0.5).astype(np.int32)

print("Grouped test hamming loss:", hamming_loss(y_test_g, test_pred))
print("Grouped test macro F1:", f1_score(y_test_g, test_pred, average="macro", zero_division=0))